# korea_fx_db_update_v1 — 원화 환율 수집·DB 적재 (일별 원천 + 월/분기 집계)

원/달러, 원/엔, 원/유로, 원/위안 등 원화 환율을 여러 소스에서 받아 `investar` 에 적재하고, 실행할 때마다 신규분만 증분 갱신한다.

| 테이블/뷰 | 층 | 키 | 용도 |
|---|---|---|---|
| `korea_fx_meta` | 메타 | pair | 통화쌍 정의, 단위, 소스 심볼, 유효범위 |
| `korea_fx_daily` | 원천 | (pair, source, date) | 소스별 일별 환율 원본 (소스 간 비교·재구성용) |
| `korea_fx_monthly` | 집계 | (pair, ym) | 월평균 / 월말 종가 / 고저 / 거래일수 / 완결여부 |
| `korea_fx_quarterly` | 집계 | (pair, yq) | 분기평균 / 분기말 종가 (재무 분기 패널 결합용) |
| `korea_fx_update_log` | 운영 | id | 실행별 수집·적재 기록 (데이터 신선도 확인) |
| `v_korea_fx_daily_best` | 뷰 | (pair, date) | 날짜별 최우선 소스 1행 (ECOS > FDR > YF) |
| `v_korea_fx_monthly_wide` / `v_korea_fx_quarterly_wide` | 뷰 | ym / yq | 통화쌍을 열로 펼친 분석용 와이드 뷰 |

**값 규약**: 모든 `close/avg_close/end_close` 는 **외화 1단위당 원화** (JPY 도 1엔 기준, 100엔 표시는 `meta.display_unit` 참조).
**집계 규약**: 월/분기 집계는 `v_korea_fx_daily_best` 와 같은 우선순위로 날짜별 1개 값을 골라 계산한다 (`CURATED_SOURCE` 로 고정 가능).

## PART 1 — 환경 · Control Panel · DB

In [1]:
import os
import sys
import getpass
import warnings
from datetime import datetime, date, timedelta
from pathlib import Path
from typing import Optional, List, Dict, Tuple

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)


def detect_project_root(max_up: int = 5) -> Optional[Path]:
    """DATA/config.py 를 가진 프로젝트 루트 탐색 (데스크톱/노트북 공통).
    1) 현재 폴더에서 상위로 max_up 단계  2) ~/PycharmProjects/* 한 단계"""
    here = Path.cwd().resolve()
    for p in [here] + list(here.parents)[:max_up]:
        if (p / "DATA" / "config.py").exists():
            return p
    base = Path.home() / "PycharmProjects"
    if base.exists():
        for p in sorted(base.iterdir()):
            if (p / "DATA" / "config.py").exists():
                return p
    return None


PROJECT_ROOT = detect_project_root()
print("home        :", Path.home())
print("PROJECT_ROOT:", PROJECT_ROOT)

home        : C:\Users\82108
PROJECT_ROOT: C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy


In [2]:
# ============================ Control Panel ============================
DB_NAME          = "investar"          # 대상 스키마 (없으면 생성)
USE_CONFIG       = True                # True: DATA/config.py 의 get_engine(get_db_info()) 로 접속
DB_HOST          = "127.0.0.1"         # USE_CONFIG=False 이거나 config 접속 실패 시 사용
DB_PORT          = 3307                # MariaDB 포트
DB_USER          = "stox7412"          # DB 사용자
DB_PASSWORD_ENV  = "INVESTAR_DB_PW"    # 비밀번호 환경변수명 (없으면 getpass 로 입력)

PAIRS            = ["USDKRW", "JPYKRW", "EURKRW", "CNYKRW"]  # 적재 통화쌍 (FX_META 키). 예: "TWDKRW" 추가 가능
SOURCES          = ["ECOS", "FDR", "YF"]  # 수집 시도 소스. ECOS 는 키가 있을 때만 수행
SOURCE_PRIORITY  = {"ECOS": 1, "FDR": 2, "YF": 3}  # 집계/뷰에서 날짜별 소스 선택 순위 (작을수록 우선)
CURATED_SOURCE   = "BEST"              # 월/분기 집계 소스: "BEST"(우선순위 결합) | "ECOS" | "FDR" | "YF"
ECOS_API_KEY_ENV = "ECOS_API_KEY"      # 한국은행 ECOS 인증키 환경변수명
ECOS_API_KEY     = "O6FJIBYZCWZF9ZFHL6F9"                  # 직접 입력 시 사용 (비우면 환경변수 조회)

START_DATE       = "2000-01-01"        # 최초 적재 시작일
LOOKBACK_DAYS    = 14                  # 증분 갱신 시 마지막 적재일 기준 N일 재수집 (수정치 반영)
FULL_RELOAD      = False               # True: START_DATE 부터 전 구간 재수집
DRY_RUN          = False               # True: DB 쓰기 없이 수집 결과만 출력
UPSERT_CHUNK     = 5000                # executemany 청크 크기
# =======================================================================

# 통화쌍 정의. 값은 모두 '외화 1단위당 원화'로 저장한다.
#   ecos_div : ECOS 원값을 1단위 기준으로 바꾸는 나눗셈 (엔화는 100엔 기준 고시 -> 100)
#   lo / hi  : 1단위 기준 유효범위. 범위 밖 값은 단위 오류로 보고 버린다.
FX_META: Dict[str, dict] = {
    "USDKRW": dict(base_ccy="USD", name_kr="원/미국달러", display_unit=1,
                   fdr="USD/KRW", yf="KRW=X",    ecos_item="0000001", ecos_div=1,   lo=700.0, hi=2500.0),
    "JPYKRW": dict(base_ccy="JPY", name_kr="원/일본엔",   display_unit=100,
                   fdr="JPY/KRW", yf="JPYKRW=X", ecos_item="0000002", ecos_div=100, lo=4.0,   hi=25.0),
    "EURKRW": dict(base_ccy="EUR", name_kr="원/유로",     display_unit=1,
                   fdr="EUR/KRW", yf="EURKRW=X", ecos_item="0000003", ecos_div=1,   lo=800.0, hi=2800.0),
    "CNYKRW": dict(base_ccy="CNY", name_kr="원/중국위안", display_unit=1,
                   fdr="CNY/KRW", yf="CNYKRW=X", ecos_item="0000053", ecos_div=1,   lo=90.0,  hi=400.0),
    "TWDKRW": dict(base_ccy="TWD", name_kr="원/대만달러", display_unit=1,
                   fdr="TWD/KRW", yf="TWDKRW=X", ecos_item=None,      ecos_div=1,   lo=20.0,  hi=80.0),
}
ECOS_STAT_CODE = "731Y001"   # 주요국통화의 대원화환율 (일별)

TODAY = pd.Timestamp.today().normalize()
if not ECOS_API_KEY:
    ECOS_API_KEY = os.environ.get(ECOS_API_KEY_ENV, "")
unknown = [p for p in PAIRS if p not in FX_META]
assert not unknown, f"FX_META 에 없는 통화쌍: {unknown}"
print("TODAY:", TODAY.date(), "| PAIRS:", PAIRS, "| ECOS key:", "있음" if ECOS_API_KEY else "없음(ECOS 생략)")

TODAY: 2026-09-29 | PAIRS: ['USDKRW', 'JPYKRW', 'EURKRW', 'CNYKRW'] | ECOS key: 있음


In [3]:
def _url_with_db(url, database: Optional[str]):
    """SQLAlchemy 1.4+/2.x URL 의 database 교체 (1.3 호환 포함)"""
    if hasattr(url, "_replace"):          # 1.4+/2.x: URL 은 NamedTuple (set() 은 None 을 무시함)
        return url._replace(database=database)
    url = __import__("copy").copy(url)    # 1.3: 가변 객체
    url.database = database
    return url


def _base_url():
    """접속 URL 결정: config 우선, 실패 시 Control Panel 값"""
    if USE_CONFIG and PROJECT_ROOT is not None:
        try:
            for p in (str(PROJECT_ROOT), str(PROJECT_ROOT / "DATA")):
                if p not in sys.path:
                    sys.path.insert(0, p)
            import config  # DATA/config.py
            eng = config.get_engine(config.get_db_info())
            print("[db] config.get_engine 사용")
            return eng.url
        except Exception as e:
            print(f"[db] config 접속 정보 사용 실패 -> Control Panel 값 사용 ({type(e).__name__}: {e})")
    from sqlalchemy.engine import URL
    pw = os.environ.get(DB_PASSWORD_ENV) or getpass.getpass(f"{DB_USER}@{DB_HOST}:{DB_PORT} 비밀번호: ")
    return URL.create("mysql+pymysql", username=DB_USER, password=pw,
                      host=DB_HOST, port=DB_PORT, query={"charset": "utf8mb4"})


def connect_and_ensure_db() -> Engine:
    base = _base_url()
    server = create_engine(_url_with_db(base, None), pool_pre_ping=True)
    with server.begin() as conn:
        exists = conn.execute(text("SELECT SCHEMA_NAME FROM information_schema.SCHEMATA WHERE SCHEMA_NAME=:d"),
                              {"d": DB_NAME}).fetchone()
        if exists is None:
            conn.execute(text(f"CREATE DATABASE `{DB_NAME}` CHARACTER SET utf8mb4 COLLATE utf8mb4_general_ci"))
            print(f"[db] 스키마 `{DB_NAME}` 생성")
        else:
            print(f"[db] 스키마 `{DB_NAME}` 존재")
    server.dispose()
    return create_engine(_url_with_db(base, DB_NAME), pool_pre_ping=True)


engine = connect_and_ensure_db()
with engine.connect() as c:
    print("[db] 접속:", c.execute(text("SELECT DATABASE(), VERSION()")).fetchone())

[db] config.get_engine 사용
[db] 스키마 `investar` 존재
[db] 접속: ('investar', '10.11.6-MariaDB')


In [4]:
TABLE_OPTS = "ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_general_ci"

DDL = [
f"""
CREATE TABLE IF NOT EXISTS korea_fx_meta (
  pair          VARCHAR(10)  NOT NULL COMMENT '통화쌍 코드 (예: USDKRW = 1 USD 당 KRW)',
  base_ccy      CHAR(3)      NOT NULL COMMENT '외화 통화코드 (ISO 4217)',
  quote_ccy     CHAR(3)      NOT NULL DEFAULT 'KRW' COMMENT '표시통화 (항상 KRW)',
  name_kr       VARCHAR(40)  NULL     COMMENT '한글명',
  value_unit    VARCHAR(40)  NOT NULL COMMENT '저장값 단위 설명 (예: KRW per 1 JPY)',
  display_unit  INT          NOT NULL DEFAULT 1 COMMENT '관행 표시 단위 (JPY=100 -> 값x100 이 100엔당 원)',
  fdr_symbol    VARCHAR(20)  NULL     COMMENT 'FinanceDataReader 심볼',
  yf_symbol     VARCHAR(20)  NULL     COMMENT 'yfinance 심볼',
  ecos_stat     VARCHAR(10)  NULL     COMMENT 'ECOS 통계표 코드',
  ecos_item     VARCHAR(10)  NULL     COMMENT 'ECOS 항목 코드',
  valid_lo      DECIMAL(18,6) NULL    COMMENT '유효범위 하한 (1단위 기준)',
  valid_hi      DECIMAL(18,6) NULL    COMMENT '유효범위 상한 (1단위 기준)',
  updated_at    DATETIME     NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
  PRIMARY KEY (pair)
) {TABLE_OPTS} COMMENT='원화 환율 통화쌍 정의';
""",
f"""
CREATE TABLE IF NOT EXISTS korea_fx_daily (
  pair        VARCHAR(10)   NOT NULL COMMENT '통화쌍 (korea_fx_meta.pair)',
  source      VARCHAR(8)    NOT NULL COMMENT '데이터 소스: ECOS(매매기준율) | FDR | YF(yfinance 종가)',
  date        DATE          NOT NULL COMMENT '거래일',
  open        DECIMAL(18,6) NULL     COMMENT '시가 (ECOS 는 NULL)',
  high        DECIMAL(18,6) NULL     COMMENT '고가 (ECOS 는 NULL)',
  low         DECIMAL(18,6) NULL     COMMENT '저가 (ECOS 는 NULL)',
  close       DECIMAL(18,6) NOT NULL COMMENT '종가 또는 매매기준율, 외화 1단위당 KRW',
  created_at  DATETIME      NOT NULL DEFAULT CURRENT_TIMESTAMP,
  updated_at  DATETIME      NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
  PRIMARY KEY (pair, source, date),
  KEY idx_fx_daily_date (date),
  KEY idx_fx_daily_pair_date (pair, date)
) {TABLE_OPTS} COMMENT='원화 환율 일별 원천 (소스별 보존)';
""",
f"""
CREATE TABLE IF NOT EXISTS korea_fx_monthly (
  pair         VARCHAR(10)   NOT NULL COMMENT '통화쌍',
  ym           DATE          NOT NULL COMMENT '해당 월 1일 (월 식별자)',
  ym_str       CHAR(7)       NOT NULL COMMENT 'YYYY-MM (korea_monthly_trade_data 결합용)',
  avg_close    DECIMAL(18,6) NOT NULL COMMENT '월평균 = 거래일 종가 단순평균',
  end_close    DECIMAL(18,6) NOT NULL COMMENT '월말 종가 = 월 마지막 거래일 값',
  end_date     DATE          NOT NULL COMMENT '월 마지막 거래일',
  first_close  DECIMAL(18,6) NOT NULL COMMENT '월 첫 거래일 값',
  high         DECIMAL(18,6) NOT NULL COMMENT '월중 최고 (일별 종가 기준)',
  low          DECIMAL(18,6) NOT NULL COMMENT '월중 최저 (일별 종가 기준)',
  n_days       SMALLINT      NOT NULL COMMENT '집계 거래일수',
  is_complete  TINYINT(1)    NOT NULL COMMENT '1=월 종료 후 집계(확정), 0=진행 중인 월',
  source_main  VARCHAR(8)    NOT NULL COMMENT '해당 월 최다 사용 소스',
  updated_at   DATETIME      NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
  PRIMARY KEY (pair, ym),
  KEY idx_fx_monthly_ym (ym),
  KEY idx_fx_monthly_ymstr (ym_str)
) {TABLE_OPTS} COMMENT='원화 환율 월별 집계 (평균/월말/고저)';
""",
f"""
CREATE TABLE IF NOT EXISTS korea_fx_quarterly (
  pair         VARCHAR(10)   NOT NULL COMMENT '통화쌍',
  yq           CHAR(6)       NOT NULL COMMENT 'YYYYQn (예: 2026Q3), 재무 분기 패널 결합키',
  q_start      DATE          NOT NULL COMMENT '분기 첫날',
  q_end        DATE          NOT NULL COMMENT '분기 마지막 날 (캘린더)',
  avg_close    DECIMAL(18,6) NOT NULL COMMENT '분기평균 = 거래일 종가 단순평균 (손익 환산 기준)',
  end_close    DECIMAL(18,6) NOT NULL COMMENT '분기말 종가 (재무상태표 환산 기준)',
  end_date     DATE          NOT NULL COMMENT '분기 마지막 거래일',
  n_days       SMALLINT      NOT NULL COMMENT '집계 거래일수',
  is_complete  TINYINT(1)    NOT NULL COMMENT '1=분기 종료 후 집계(확정), 0=진행 중',
  updated_at   DATETIME      NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
  PRIMARY KEY (pair, yq),
  KEY idx_fx_quarterly_yq (yq)
) {TABLE_OPTS} COMMENT='원화 환율 분기 집계 (평균/분기말)';
""",
f"""
CREATE TABLE IF NOT EXISTS korea_fx_update_log (
  id             BIGINT       NOT NULL AUTO_INCREMENT,
  run_id         VARCHAR(20)  NOT NULL COMMENT '실행 식별자 YYYYMMDDHHMMSS',
  run_at         DATETIME     NOT NULL DEFAULT CURRENT_TIMESTAMP,
  pair           VARCHAR(10)  NOT NULL,
  source         VARCHAR(8)   NOT NULL,
  fetch_start    DATE         NULL,
  fetch_end      DATE         NULL,
  rows_fetched   INT          NOT NULL DEFAULT 0,
  rows_dropped   INT          NOT NULL DEFAULT 0 COMMENT '유효범위/결측 검증 탈락',
  rows_affected  INT          NOT NULL DEFAULT 0 COMMENT 'MySQL affected rows (insert=1, update=2, 동일=0)',
  max_date       DATE         NULL     COMMENT '적재 후 해당 pair/source 최신일',
  status         VARCHAR(10)  NOT NULL COMMENT 'OK | EMPTY | SKIP | ERROR | DRYRUN',
  message        VARCHAR(500) NULL,
  PRIMARY KEY (id),
  KEY idx_fx_log_run (run_id),
  KEY idx_fx_log_pair (pair, source, run_at)
) {TABLE_OPTS} COMMENT='환율 수집 실행 기록';
""",
]


def priority_case(col: str = "source") -> str:
    parts = " ".join(f"WHEN '{s}' THEN {p}" for s, p in SOURCE_PRIORITY.items())
    return f"CASE {col} {parts} ELSE 99 END"


def create_wide_views(conn, pairs: List[str]) -> None:
    """통화쌍을 열로 펼친 와이드 뷰 (월/분기). 열 이름: usdkrw_avg, usdkrw_end ..."""
    def cols(avg: str = "avg_close", end: str = "end_close") -> str:
        out = []
        for p in pairs:
            k = p.lower()
            out.append(f"MAX(CASE WHEN pair='{p}' THEN {avg} END) AS {k}_avg")
            out.append(f"MAX(CASE WHEN pair='{p}' THEN {end} END) AS {k}_end")
        return ",\n  ".join(out)
    conn.execute(text(f"""CREATE OR REPLACE VIEW v_korea_fx_monthly_wide AS
        SELECT ym, ym_str, MIN(is_complete) AS is_complete,
          {cols()}
        FROM korea_fx_monthly GROUP BY ym, ym_str"""))
    conn.execute(text(f"""CREATE OR REPLACE VIEW v_korea_fx_quarterly_wide AS
        SELECT yq, MIN(is_complete) AS is_complete,
          {cols()}
        FROM korea_fx_quarterly GROUP BY yq"""))


def create_schema(eng: Engine) -> None:
    with eng.begin() as conn:
        for ddl in DDL:
            conn.execute(text(ddl))
        conn.execute(text(f"""
            CREATE OR REPLACE VIEW v_korea_fx_daily_best AS
            SELECT pair, date, source, open, high, low, close
            FROM (
                SELECT d.*, ROW_NUMBER() OVER (PARTITION BY d.pair, d.date
                                               ORDER BY {priority_case('d.source')}) AS rn
                FROM korea_fx_daily d
            ) t
            WHERE rn = 1
        """))
        rows = [dict(pair=k, base_ccy=v["base_ccy"], quote_ccy="KRW", name_kr=v["name_kr"],
                     value_unit=f"KRW per 1 {v['base_ccy']}", display_unit=v["display_unit"],
                     fdr_symbol=v["fdr"], yf_symbol=v["yf"],
                     ecos_stat=ECOS_STAT_CODE if v["ecos_item"] else None, ecos_item=v["ecos_item"],
                     valid_lo=v["lo"], valid_hi=v["hi"]) for k, v in FX_META.items()]
        cols = list(rows[0].keys())
        conn.execute(text(
            f"INSERT INTO korea_fx_meta ({', '.join(cols)}) VALUES ({', '.join(':' + c for c in cols)}) "
            f"ON DUPLICATE KEY UPDATE " + ", ".join(f"{c}=VALUES({c})" for c in cols if c != "pair")), rows)
        create_wide_views(conn, sorted(PAIRS))   # 데이터 적재 전에도 뷰가 존재하도록 (적재 후 실제 pair 로 재생성)
    print("[schema] 테이블 5개 + 뷰 3개 준비 완료, meta", len(FX_META), "행")


create_schema(engine)

[schema] 테이블 5개 + 뷰 3개 준비 완료, meta 5 행


## PART 2 — 엔진 (수집 · 검증 · 적재 · 집계)

In [5]:
COLS = ["date", "open", "high", "low", "close"]


def _empty() -> pd.DataFrame:
    return pd.DataFrame(columns=COLS)


def _flatten(raw: pd.DataFrame) -> pd.DataFrame:
    """FDR / yfinance 결과를 date/open/high/low/close 로 정규화"""
    if raw is None or len(raw) == 0:
        return _empty()
    df = raw.copy()
    if isinstance(df.columns, pd.MultiIndex):          # yfinance 0.2.x 단일티커 MultiIndex
        df.columns = df.columns.get_level_values(0)
    df.columns = [str(c).strip().lower() for c in df.columns]
    df = df.reset_index()
    df = df.rename(columns={df.columns[0]: "date"})
    for c in COLS[1:]:
        if c not in df.columns:
            df[c] = np.nan
    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None).dt.normalize()
    return df[COLS]


def fetch_fdr(pair: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    import FinanceDataReader as fdr
    return _flatten(fdr.DataReader(FX_META[pair]["fdr"], start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d")))


def fetch_yf(pair: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    import yfinance as yf
    raw = yf.download(FX_META[pair]["yf"], start=start.strftime("%Y-%m-%d"),
                      end=(end + timedelta(days=1)).strftime("%Y-%m-%d"),   # end 미포함 보정
                      progress=False, auto_adjust=False, threads=False)
    return _flatten(raw)


def fetch_ecos(pair: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    import requests
    meta = FX_META[pair]
    url = (f"https://ecos.bok.or.kr/api/StatisticSearch/{ECOS_API_KEY}/json/kr/1/100000/"
           f"{ECOS_STAT_CODE}/D/{start:%Y%m%d}/{end:%Y%m%d}/{meta['ecos_item']}")
    js = requests.get(url, timeout=30).json()
    if "StatisticSearch" not in js:
        res = js.get("RESULT", {})
        if str(res.get("CODE", "")).startswith("INFO-200"):     # 해당 데이터 없음
            return _empty()
        raise RuntimeError(f"ECOS 응답 오류: {res}")
    rows = js["StatisticSearch"].get("row", [])
    df = pd.DataFrame(rows)
    if df.empty:
        return _empty()
    out = pd.DataFrame({
        "date": pd.to_datetime(df["TIME"], format="%Y%m%d"),
        "close": pd.to_numeric(df["DATA_VALUE"].astype(str).str.replace(",", ""), errors="coerce") / meta["ecos_div"],
    })
    for c in ("open", "high", "low"):
        out[c] = np.nan
    return out[COLS]


FETCHERS = {"ECOS": fetch_ecos, "FDR": fetch_fdr, "YF": fetch_yf}
SYMBOL_KEY = {"ECOS": "ecos_item", "FDR": "fdr", "YF": "yf"}


def validate(pair: str, df: pd.DataFrame) -> Tuple[pd.DataFrame, int]:
    """결측/비양수/유효범위 밖/미래일 제거, 날짜 중복 제거. 반환: (정제본, 탈락 수)"""
    n0 = len(df)
    if n0 == 0:
        return df, 0
    lo, hi = FX_META[pair]["lo"], FX_META[pair]["hi"]
    d = df.copy()
    for c in COLS[1:]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    bad_range = (d["close"] < lo) | (d["close"] > hi)
    if bad_range.any():
        s = d.loc[bad_range, "close"]
        print(f"   [warn] {pair}: 유효범위[{lo},{hi}] 밖 {bad_range.sum()}행 제거 (예: {s.iloc[0]:.4f}) — 단위 확인 필요")
    # OHLC 는 종가와 같은 범위 검증, 벗어나면 NULL
    for c in ("open", "high", "low"):
        d.loc[(d[c] < lo) | (d[c] > hi), c] = np.nan
    d = d[d["close"].notna() & (d["close"] > 0) & ~bad_range & (d["date"] <= TODAY)]
    d = d.sort_values("date").drop_duplicates("date", keep="last")
    return d.reset_index(drop=True), n0 - len(d)

In [6]:
def _records(df: pd.DataFrame) -> List[dict]:
    out = df.astype(object).where(pd.notna(df), None).to_dict("records")
    for r in out:
        for k, v in r.items():
            if isinstance(v, pd.Timestamp):
                r[k] = v.date()
            elif isinstance(v, (np.floating, np.integer)):
                r[k] = v.item()
    return out


def upsert(conn, table: str, df: pd.DataFrame, key_cols: List[str]) -> int:
    """INSERT ... ON DUPLICATE KEY UPDATE (키 외 컬럼 갱신). 반환: affected rows 합"""
    if df is None or len(df) == 0:
        return 0
    cols = list(df.columns)
    upd = ", ".join(f"`{c}`=VALUES(`{c}`)" for c in cols if c not in key_cols)
    sql = text(f"INSERT INTO `{table}` ({', '.join('`' + c + '`' for c in cols)}) "
               f"VALUES ({', '.join(':' + c for c in cols)}) ON DUPLICATE KEY UPDATE {upd}")
    recs, n = _records(df), 0
    for i in range(0, len(recs), UPSERT_CHUNK):
        res = conn.execute(sql, recs[i:i + UPSERT_CHUNK])
        n += max(res.rowcount, 0)
    return n


def query_df(sql: str, params: Optional[dict] = None) -> pd.DataFrame:
    """pandas/SQLAlchemy 버전 조합과 무관하게 동작하는 조회"""
    with engine.connect() as conn:
        res = conn.execute(text(sql), params or {})
        return pd.DataFrame(res.fetchall(), columns=list(res.keys()))


def last_date(pair: str, source: str) -> Optional[pd.Timestamp]:
    df = query_df("SELECT MAX(date) AS d FROM korea_fx_daily WHERE pair=:p AND source=:s", {"p": pair, "s": source})
    v = df.iloc[0, 0] if len(df) else None
    return pd.Timestamp(v) if v is not None else None


def write_log(conn, **kw) -> None:
    kw.setdefault("message", None)
    kw["message"] = (kw["message"] or "")[:500] or None
    cols = list(kw.keys())
    conn.execute(text(f"INSERT INTO korea_fx_update_log ({', '.join(cols)}) VALUES ({', '.join(':' + c for c in cols)})"), kw)

In [7]:
def load_curated_daily(pairs: List[str]) -> pd.DataFrame:
    """집계용 일별 계열: CURATED_SOURCE='BEST' 면 날짜별 우선순위 1개, 아니면 해당 소스만"""
    ph = ", ".join(f":p{i}" for i in range(len(pairs)))
    prm = {f"p{i}": p for i, p in enumerate(pairs)}
    if CURATED_SOURCE == "BEST":
        sql = f"SELECT pair, date, source, close FROM v_korea_fx_daily_best WHERE pair IN ({ph})"
    else:
        sql = f"SELECT pair, date, source, close FROM korea_fx_daily WHERE pair IN ({ph}) AND source=:src"
        prm["src"] = CURATED_SOURCE
    df = query_df(sql, prm)
    if df.empty:
        return df
    df["date"] = pd.to_datetime(df["date"])
    df["close"] = df["close"].astype(float)
    return df.sort_values(["pair", "date"]).reset_index(drop=True)


def build_monthly(d: pd.DataFrame) -> pd.DataFrame:
    d = d.assign(ym=d["date"].dt.to_period("M"))
    g = d.groupby(["pair", "ym"], sort=True)
    m = g.agg(avg_close=("close", "mean"), end_close=("close", "last"), end_date=("date", "max"),
              first_close=("close", "first"), high=("close", "max"), low=("close", "min"),
              n_days=("close", "size"),
              source_main=("source", lambda s: s.value_counts().index[0])).reset_index()
    m["is_complete"] = (m["ym"].dt.end_time.dt.normalize() < TODAY).astype(int)
    m["ym_str"] = m["ym"].astype(str)
    m["ym"] = m["ym"].dt.start_time
    return m[["pair", "ym", "ym_str", "avg_close", "end_close", "end_date", "first_close",
              "high", "low", "n_days", "is_complete", "source_main"]]


def build_quarterly(d: pd.DataFrame) -> pd.DataFrame:
    d = d.assign(q=d["date"].dt.to_period("Q"))
    q = d.groupby(["pair", "q"], sort=True).agg(
        avg_close=("close", "mean"), end_close=("close", "last"),
        end_date=("date", "max"), n_days=("close", "size")).reset_index()
    q["yq"] = q["q"].astype(str)
    q["q_start"] = q["q"].dt.start_time
    q["q_end"] = q["q"].dt.end_time.dt.normalize()
    q["is_complete"] = (q["q_end"] < TODAY).astype(int)
    return q[["pair", "yq", "q_start", "q_end", "avg_close", "end_close", "end_date", "n_days", "is_complete"]]


def rebuild_curated(pairs: List[str]) -> Tuple[int, int]:
    """집계층 재계산 (일별 전 구간 기준, 소량이라 매 실행 전량 재계산 후 upsert)"""
    d = load_curated_daily(pairs)
    if d.empty:
        print("[curated] 일별 데이터 없음")
        return 0, 0
    m, q = build_monthly(d), build_quarterly(d)
    with engine.begin() as conn:
        nm = upsert(conn, "korea_fx_monthly", m, ["pair", "ym"])
        nq = upsert(conn, "korea_fx_quarterly", q, ["pair", "yq"])
        all_pairs = [r[0] for r in conn.execute(text("SELECT DISTINCT pair FROM korea_fx_monthly ORDER BY pair"))]
        create_wide_views(conn, all_pairs)
    print(f"[curated] monthly {len(m)}행(affected {nm}), quarterly {len(q)}행(affected {nq}), 와이드뷰 열: {all_pairs}")
    return nm, nq

## PART 3 — 실행 (증분 갱신)

In [8]:
def run_update() -> pd.DataFrame:
    run_id = datetime.now().strftime("%Y%m%d%H%M%S")
    logs = []
    for pair in PAIRS:
        for src in SOURCES:
            rec = dict(run_id=run_id, pair=pair, source=src)
            if FX_META[pair].get(SYMBOL_KEY[src]) is None:
                logs.append({**rec, "status": "SKIP", "message": "소스 심볼 없음"}); continue
            if src == "ECOS" and not ECOS_API_KEY:
                logs.append({**rec, "status": "SKIP", "message": "ECOS 키 없음"}); continue

            last = None if FULL_RELOAD else last_date(pair, src)
            start = pd.Timestamp(START_DATE) if last is None else max(pd.Timestamp(START_DATE), last - timedelta(days=LOOKBACK_DAYS))
            rec.update(fetch_start=start.date(), fetch_end=TODAY.date())
            try:
                raw = FETCHERS[src](pair, start, TODAY)
                clean, dropped = validate(pair, raw)
                rec.update(rows_fetched=len(raw), rows_dropped=dropped)
                if clean.empty:
                    rec.update(status="EMPTY", message="수집 0행 또는 전부 탈락")
                elif DRY_RUN:
                    rec.update(status="DRYRUN", max_date=clean["date"].max().date())
                else:
                    with engine.begin() as conn:
                        n = upsert(conn, "korea_fx_daily", clean.assign(pair=pair, source=src)[["pair", "source"] + COLS],
                                   ["pair", "source", "date"])
                    rec.update(status="OK", rows_affected=n, max_date=last_date(pair, src).date())
            except Exception as e:
                rec.update(status="ERROR", message=f"{type(e).__name__}: {e}")
            logs.append(rec)
            print(f"  {pair:7s} {src:5s} {rec.get('status'):6s} "
                  f"fetched={rec.get('rows_fetched', 0):5d} dropped={rec.get('rows_dropped', 0):3d} "
                  f"affected={rec.get('rows_affected', 0):5d} max={rec.get('max_date')} {rec.get('message') or ''}")

    log_df = pd.DataFrame(logs)
    if not DRY_RUN:
        with engine.begin() as conn:
            for r in logs:
                write_log(conn, **{k: v for k, v in r.items() if v is not None and not (isinstance(v, float) and np.isnan(v))})
        rebuild_curated(PAIRS)

    ok_pairs = set(log_df.loc[log_df["status"].isin(["OK", "DRYRUN"]), "pair"])
    missing = [p for p in PAIRS if p not in ok_pairs]
    if missing:
        print(f"[warn] 이번 실행에서 어떤 소스도 성공하지 못한 통화쌍: {missing}")
    return log_df


print(f"=== 환율 갱신 시작 {datetime.now():%Y-%m-%d %H:%M:%S} | FULL_RELOAD={FULL_RELOAD} DRY_RUN={DRY_RUN} ===")
LOG = run_update()

=== 환율 갱신 시작 2026-09-29 11:04:45 | FULL_RELOAD=False DRY_RUN=False ===
  USDKRW  ECOS  OK     fetched= 6926 dropped=  0 affected= 6926 max=2026-09-29 
  USDKRW  FDR   OK     fetched= 5956 dropped= 36 affected= 5920 max=2026-09-27 
  USDKRW  YF    OK     fetched= 5921 dropped=  0 affected= 5921 max=2026-09-29 
  JPYKRW  ECOS  OK     fetched= 6926 dropped=  0 affected= 6926 max=2026-09-29 
  JPYKRW  FDR   OK     fetched= 5956 dropped= 36 affected= 5920 max=2026-09-27 
  JPYKRW  YF    OK     fetched= 5921 dropped=  0 affected= 5921 max=2026-09-29 
  EURKRW  ECOS  OK     fetched= 6926 dropped=  0 affected= 6926 max=2026-09-29 
  EURKRW  FDR   OK     fetched= 5956 dropped= 20 affected= 5936 max=2026-09-27 
  EURKRW  YF    OK     fetched= 5937 dropped=  0 affected= 5937 max=2026-09-29 
  CNYKRW  ECOS  OK     fetched= 2643 dropped=  0 affected= 2643 max=2026-09-29 
  CNYKRW  FDR   ERROR  fetched=    0 dropped=  0 affected=    0 max=None KeyError: 'timestamp'
  CNYKRW  YF    OK     fetched=   

## PART 4 — 점검 (신선도 · 소스 간 괴리 · 최근값)

In [9]:
print("■ 소스별 적재 현황")
display(query_df("""
    SELECT pair, source, COUNT(*) AS n, MIN(date) AS first_date, MAX(date) AS last_date
    FROM korea_fx_daily GROUP BY pair, source ORDER BY pair, source"""))

print("■ 소스 간 괴리 (같은 날짜 종가, 기준=우선순위 1위 소스와 비교, 최근 250거래일)")
d_all = query_df("SELECT pair, source, date, close FROM korea_fx_daily WHERE date >= :d",
                 {"d": (TODAY - timedelta(days=365)).date()})
if not d_all.empty:
    d_all["close"] = d_all["close"].astype(float)
    wide = d_all.pivot_table(index=["pair", "date"], columns="source", values="close")
    rows = []
    for pair, g in wide.groupby(level=0):
        srcs = sorted([s for s in g.columns if g[s].notna().any()], key=lambda s: SOURCE_PRIORITY.get(s, 99))
        for s in srcs[1:]:
            both = g[[srcs[0], s]].dropna()
            if len(both):
                pct = (both[s] / both[srcs[0]] - 1).abs() * 100
                rows.append(dict(pair=pair, ref=srcs[0], other=s, n=len(both),
                                 median_abs_pct=round(pct.median(), 3), max_abs_pct=round(pct.max(), 3)))
    display(pd.DataFrame(rows) if rows else "비교 가능한 소스 쌍 없음 (소스 1개)")

print("■ 최근 월별 (와이드)")
display(query_df("SELECT * FROM v_korea_fx_monthly_wide ORDER BY ym DESC LIMIT 6"))

print("■ 최근 분기 원/달러 (분기평균 QoQ / YoY)")
qd = query_df("SELECT yq, avg_close, end_close, n_days, is_complete FROM korea_fx_quarterly "
              "WHERE pair='USDKRW' ORDER BY yq")
if not qd.empty:
    qd[["avg_close", "end_close"]] = qd[["avg_close", "end_close"]].astype(float)
    qd["avg_qoq_pct"] = (qd["avg_close"].pct_change() * 100).round(2)
    qd["avg_yoy_pct"] = (qd["avg_close"].pct_change(4) * 100).round(2)
    display(qd.tail(6))

print("■ 최근 실행 로그")
display(query_df("SELECT run_id, pair, source, status, rows_fetched, rows_affected, max_date, message "
                 "FROM korea_fx_update_log ORDER BY id DESC LIMIT 20"))

■ 소스별 적재 현황


,pair,source,n,first_date,last_date
0,CNYKRW,ECOS,2643,2016-01-04,2026-09-29
1,CNYKRW,YF,1,2026-09-29,2026-09-29
2,EURKRW,ECOS,6926,2000-01-04,2026-09-29
3,EURKRW,FDR,5936,2003-12-01,2026-09-27
4,EURKRW,YF,5937,2003-12-01,2026-09-29
5,JPYKRW,ECOS,6926,2000-01-04,2026-09-29
6,JPYKRW,FDR,5920,2003-12-01,2026-09-27
7,JPYKRW,YF,5921,2003-12-01,2026-09-29
8,USDKRW,ECOS,6926,2000-01-04,2026-09-29
9,USDKRW,FDR,5920,2003-12-01,2026-09-27


■ 소스 간 괴리 (같은 날짜 종가, 기준=우선순위 1위 소스와 비교, 최근 250거래일)


,pair,ref,other,n,median_abs_pct,max_abs_pct
0,CNYKRW,ECOS,YF,1,0.173,0.173
1,EURKRW,ECOS,FDR,215,0.353,2.431
2,EURKRW,ECOS,YF,243,0.280,3.211
3,JPYKRW,ECOS,FDR,215,0.330,2.539
4,JPYKRW,ECOS,YF,243,0.221,2.067
5,USDKRW,ECOS,FDR,215,0.350,2.452
6,USDKRW,ECOS,YF,243,0.230,1.917


■ 최근 월별 (와이드)


,ym,ym_str,is_complete,cnykrw_avg,cnykrw_end,eurkrw_avg,eurkrw_end,jpykrw_avg,jpykrw_end,usdkrw_avg,usdkrw_end
0,2026-09-01,2026-09,0,202.602632,202.200000,1566.738800,1546.460000,8.691957,8.639000,1359.411602,1360.000000
1,2026-08-01,2026-08,1,208.686000,205.050000,1626.727690,1594.470000,8.854991,8.592700,1406.188073,1376.500000
2,2026-07-01,2026-07,1,220.942727,213.560000,1706.739626,1661.160000,9.208378,9.021300,1496.189999,1441.100000
3,2026-06-01,2026-06,1,225.506667,226.970000,1758.299231,1761.090000,9.509451,9.519800,1528.229227,1541.500000
4,2026-05-01,2026-05,1,219.110556,222.180000,1741.042696,1755.900024,9.420122,9.455424,1489.759999,1506.839966
5,2026-04-01,2026-04,1,217.406818,215.900000,1737.556157,1723.570000,9.332136,9.205800,1486.687693,1476.100000


■ 최근 분기 원/달러 (분기평균 QoQ / YoY)


,yq,avg_close,end_close,n_days,is_complete,avg_qoq_pct,avg_yoy_pct
101,2025Q2,1400.908442,1356.4,77,1,-3.51,2.22
102,2025Q3,1385.657468,1402.2,79,1,-1.09,2.14
103,2025Q4,1446.876178,1434.9,68,1,4.42,3.86
104,2026Q1,1464.216250,1513.4,64,1,1.20,0.85
105,2026Q2,1501.558973,1541.5,78,1,2.55,7.18
106,2026Q3,1422.350127,1360.0,78,0,-5.28,2.65


■ 최근 실행 로그


,run_id,pair,source,status,rows_fetched,rows_affected,max_date,message
0,20260929110445,CNYKRW,YF,OK,1,1,2026-09-29,None
1,20260929110445,CNYKRW,FDR,ERROR,0,0,None,KeyError: 'timestamp'
2,20260929110445,CNYKRW,ECOS,OK,2643,2643,2026-09-29,None
3,20260929110445,EURKRW,YF,OK,5937,5937,2026-09-29,None
4,20260929110445,EURKRW,FDR,OK,5956,5936,2026-09-27,None
5,20260929110445,EURKRW,ECOS,OK,6926,6926,2026-09-29,None
6,20260929110445,JPYKRW,YF,OK,5921,5921,2026-09-29,None
7,20260929110445,JPYKRW,FDR,OK,5956,5920,2026-09-27,None
8,20260929110445,JPYKRW,ECOS,OK,6926,6926,2026-09-29,None
9,20260929110445,USDKRW,YF,OK,5921,5921,2026-09-29,None


## 부록 — 다른 분석에서 쓰는 법 (Claude Code 참고)

스키마 설명은 DB 에도 들어 있다: `SHOW FULL COLUMNS FROM korea_fx_monthly;` 로 컬럼별 COMMENT 확인.

```sql
-- 분기평균 원/달러 (재무 분기 패널의 fx 열)
SELECT yq, avg_close AS fx, is_complete FROM korea_fx_quarterly WHERE pair='USDKRW' ORDER BY yq;

-- 월별 수출(USD) 원화 환산: ym_str 로 결합
SELECT t.*, f.avg_close AS usdkrw
FROM korea_monthly_trade_data t
JOIN korea_fx_monthly f ON f.pair='USDKRW' AND f.ym_str = <t 의 YYYY-MM 표현식>;

-- 모든 통화쌍 월평균·월말 한 번에
SELECT * FROM v_korea_fx_monthly_wide WHERE ym >= '2020-01-01' ORDER BY ym;

-- 100엔당 원 (관행 표시)
SELECT ym_str, avg_close * 100 AS krw_per_100jpy FROM korea_fx_monthly WHERE pair='JPYKRW';

-- 데이터 신선도
SELECT pair, source, MAX(date) FROM korea_fx_daily GROUP BY pair, source;
```

```python
# 분기 패널 결합 예 (hynix_nowcast_model_v1 의 fx 열)
fx = query_df("SELECT yq, avg_close FROM korea_fx_quarterly WHERE pair='USDKRW'")
fx.index = pd.PeriodIndex(fx["yq"], freq="Q")
panel["fx"] = fx["avg_close"].astype(float)
```

주의
- 진행 중인 월/분기(`is_complete=0`)는 그때까지의 거래일 평균이다. 확정치만 쓸 때는 `is_complete=1` 로 거른다.
- `BEST` 집계는 날짜별로 소스가 섞일 수 있다(ECOS 매매기준율 vs 시장 종가, 괴리는 PART 4 표 참조). 단일 소스 일관성이 필요하면 `CURATED_SOURCE` 를 고정하고 재실행한다.
- ECOS 항목코드(`ecos_item`)는 FX_META 에서 관리한다. 값이 유효범위를 벗어나면 적재 전에 버리고 경고를 출력하므로 단위 착오는 로그에서 드러난다.